# 02 · Normalización

**Normalizar** es uniformizar el texto para reducir variaciones que no aportan
significado:

1. *lowercasing* → todo a minúsculas
2. eliminación de puntuación y símbolos
3. *stopwords removal* → quitar palabras muy comunes
4. *stemming* o *lematización* → llevar cada palabra a su forma base

Corpus: los **testimonios / comentarios** de clientes de la tienda-virtual, en
`datos/comentarios.csv`.

In [1]:
from pathlib import Path

import pandas as pd

# El texto ya minado de la tienda-virtual está copiado en la carpeta datos/ de
# este mismo proyecto:
#   datos/resenas_entrega.csv   reseñas de entrega (post_compra)
#   datos/comentarios.csv       testimonios / comentarios de clientes
#   datos/productos.csv         catálogo con la descripción de cada producto
DATOS = Path("../datos")


def cargar(nombre, **kwargs):
    """Lee un CSV de la carpeta datos/ y lo devuelve como DataFrame."""
    ruta = DATOS / nombre
    if not ruta.exists():
        raise FileNotFoundError(f"No se encontró {ruta.resolve()}")
    print(f"Leyendo {ruta}  ({ruta.stat().st_size / 1024:.1f} KB)")
    return pd.read_csv(ruta, **kwargs)

In [2]:
comentarios = cargar("comentarios.csv")
corpus = comentarios["texto"].dropna().astype(str).tolist()
print(f"{len(corpus)} comentarios")
comentarios[["calificacion", "texto"]].head(5)

Leyendo ../datos/comentarios.csv  (22.4 KB)
135 comentarios


,calificacion,texto
0,5,El proceso de facturacion para empresa fue mas...
1,3,"En general cumple lo que promete, ni mejor ni ..."
2,4,"Buena tienda en general, seguire comprando aqu..."
3,3,El envio internacional tardo mas dias de los i...
4,3,El proceso de devolucion fue sencillo de inici...


## Paso a paso sobre un comentario

In [3]:
import re

original = corpus[3]
print("0. original          :", original)

minuscula = original.lower()
print("1. minúsculas        :", minuscula)

sin_signos = re.sub(r"[^\w\s]", " ", minuscula)
sin_signos = re.sub(r"\s+", " ", sin_signos).strip()
print("2. sin puntuación    :", sin_signos)

0. original          : El envio internacional tardo mas dias de los indicados originalmente.
1. minúsculas        : el envio internacional tardo mas dias de los indicados originalmente.
2. sin puntuación    : el envio internacional tardo mas dias de los indicados originalmente


## Stopwords

NLTK trae una lista de *stopwords* en español. Quitarlas reduce el ruido, aunque
en tareas de estilo o búsqueda exacta de frases conviene conservarlas.

In [4]:
import nltk

nltk.download("stopwords", quiet=True)
stop_es = set(nltk.corpus.stopwords.words("spanish"))
print(f"{len(stop_es)} stopwords en español. Ejemplos: {sorted(list(stop_es))[:15]}")

tokens = sin_signos.split()
sin_stop = [t for t in tokens if t not in stop_es]
quitadas = [t for t in tokens if t in stop_es]

print("\n3. sin stopwords     :", sin_stop)
print("   (se quitaron)      :", quitadas)

313 stopwords en español. Ejemplos: ['a', 'al', 'algo', 'algunas', 'algunos', 'ante', 'antes', 'como', 'con', 'contra', 'cual', 'cuando', 'de', 'del', 'desde']

3. sin stopwords     : ['envio', 'internacional', 'tardo', 'mas', 'dias', 'indicados', 'originalmente']
   (se quitaron)      : ['el', 'de', 'los']


## Stemming (Snowball) vs Lematización (spaCy)

- **Stemming**: recorta la palabra a una *raíz* con reglas mecánicas. Rápido, pero
  la raíz puede no ser una palabra real (`entrega` → `entreg`).
- **Lematización**: devuelve el *lema* (forma de diccionario) usando contexto y
  categoría gramatical (`entregados` → `entregar`).

In [5]:
import spacy
from nltk.stem import SnowballStemmer

nlp = spacy.load("es_core_news_sm")
stemmer = SnowballStemmer("spanish")

palabras = ["llegó", "llegaron", "llegando", "entrega", "entregado", "entregaron",
            "rápido", "rápidamente", "comprando", "compré", "productos", "devolución"]

comparacion = pd.DataFrame(
    {
        "palabra": palabras,
        "stem (Snowball)": [stemmer.stem(p) for p in palabras],
        "lema (spaCy)": [nlp(p)[0].lemma_ for p in palabras],
    }
)
comparacion

,palabra,stem (Snowball),lema (spaCy)
0,llegó,lleg,llegar
1,llegaron,lleg,llegar
2,llegando,lleg,llegar
3,entrega,entreg,entrega
4,entregado,entreg,entregado
5,entregaron,entreg,entregar
6,rápido,rap,rápido
7,rápidamente,rapid,rápidamente
8,comprando,compr,comprar
9,compré,compr,comprar


## Pipeline de normalización completo

Combinamos todos los pasos en una función reutilizable. La usaremos también en los
notebooks de Bag of Words, TF-IDF y minería de texto.

In [6]:
import re
import unicodedata

import nltk
import spacy

nlp = spacy.load("es_core_news_sm")
STOPWORDS = set(nltk.corpus.stopwords.words("spanish"))


def _limpiar(texto):
    # NFKC junta tildes combinantes sueltas; luego minúsculas y solo letras/espacios
    texto = unicodedata.normalize("NFKC", str(texto)).lower()
    return re.sub(r"[^\w\s]", " ", texto)


def _lemas(doc):
    return [
        t.lemma_.lower()
        for t in doc
        if t.is_alpha and not t.is_stop and t.lemma_.lower() not in STOPWORDS and len(t.lemma_) > 2
    ]


def normalizar(texto):
    """minúsculas -> sin signos -> sin stopwords -> lematizado. Devuelve un str."""
    return " ".join(_lemas(nlp(_limpiar(texto))))


def normalizar_muchos(textos):
    """Igual que normalizar() pero en lote con nlp.pipe (más rápido para un corpus)."""
    return [" ".join(_lemas(doc)) for doc in nlp.pipe([_limpiar(t) for t in textos], batch_size=64)]

In [7]:
muestra = pd.DataFrame(
    {
        "original": corpus[:6],
        "normalizado": normalizar_muchos(corpus[:6]),
    }
)
pd.set_option("display.max_colwidth", 90)
muestra

,original,normalizado
0,El proceso de facturacion para empresa fue mas simple de lo que esperaba.,proceso facturacion empresa simple esperar
1,"En general cumple lo que promete, ni mejor ni peor que otras tiendas de tecnologia que...",general cumplir prometer tienda tecnologia probar
2,"Buena tienda en general, seguire comprando aqui mis proximos equipos.",tienda general seguire comprar proximo equipo
3,El envio internacional tardo mas dias de los indicados originalmente.,envio internacional tardo indicado originalmente
4,El proceso de devolucion fue sencillo de iniciar aunque tomo unos dias en confirmarse.,proceso devolucion sencillo iniciar tomo confirmar él
5,Muy buena atencion personalizada cuando pregunte por disponibilidad de un modelo agotado.,atencion personalizado pregunte disponibilidad modelo agotado


---
**Siguiente:** `03_ngram.ipynb` — antes de descartar el orden por completo (Bag
of Words), veamos cómo los n-gramas capturan el contexto local.